# 03 · One-stage anchor-based: RetinaNet (torchvision `retinanet_resnet50_fpn_v2`)
Same protocol as 02 (frames, val, metric, optimiser, 4 epochs, ignore regions masked in the focal loss).
Runs: `retinanet_baseline` defaults 800/1333 (~1.7 h) → `retinanet_anchors` base sizes 16-256 from the EDA (~1.7 h) →
`retinanet_assign` + IoU thresholds 0.4 / 0.3 (~1.7 h) → `retinanet_tuned` + 1024/1824 input (~3.5 h, writes a submission).

In [1]:
# Paths: auto-detected for Colab, Kaggle and a local clone. Override by hand if your layout differs.
import os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    ENV = "colab"
    from google.colab import drive
    drive.mount("/content/drive")                                   # outputs survive disconnects
    DATA_ROOT = Path("/content/data")                               # folder with test_images.csv
    REPO_DIR, WORK_DIR = Path("/content/Video-Detection-HW2"), Path("/content/drive/MyDrive/hw2_outputs")
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):                      # Colab also has /kaggle/input -> check env var
    ENV = "kaggle"
    DATA_ROOT = next(Path("/kaggle/input").glob("**/test_images.csv")).parent  # competition data
    REPO_DIR, WORK_DIR = Path("/kaggle/working/Video-Detection-HW2"), Path("/kaggle/working/outputs")
else:
    ENV = "local"
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_ROOT, WORK_DIR = REPO_DIR / "data", REPO_DIR / "outputs"
assert (DATA_ROOT / "test_images.csv").exists(), f"test_images.csv not found in {DATA_ROOT}"
print("env:", ENV, "| DATA_ROOT:", DATA_ROOT, "| REPO_DIR:", REPO_DIR, "| WORK_DIR:", WORK_DIR)

env: local | DATA_ROOT: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/data | REPO_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2 | WORK_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs


In [2]:
# Get the code + pinned dependencies on Kaggle / Colab (needs Internet). Locally nothing happens.
import subprocess
REPO_URL = "https://github.com/ystacy-ab/Video-Detection-HW2.git"
if ENV != "local":
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

In [3]:
# Make `src/` importable and print library versions (reproducibility)
sys.path.insert(0, str(Path(REPO_DIR).resolve()))
from src.utils import load_config, load_exp_config, seed_everything, print_versions, ensure_dir
print_versions()

python          3.12.6
numpy           2.4.6
torch           2.11.0
torchvision     0.26.0
ultralytics     8.4.208
faster_coco_eval 1.8.0
pycocotools     installed
ensemble_boxes  installed
sahi            0.14.0
cv2             5.0.0
wandb           0.31.0
device          CPU only


In [4]:
# Shared config + this notebook's experiment configs, seeds, output folders
cfg = load_config(Path(REPO_DIR) / "configs" / "base.yaml")
seed_everything(cfg["seed"])
DATA_ROOT, WORK_DIR = Path(DATA_ROOT), ensure_dir(WORK_DIR)
RES_DIR, FIG_DIR = Path(REPO_DIR) / "results", ensure_dir(Path(REPO_DIR) / "results" / "figures")
GT_VAL = DATA_ROOT / "val" / "annotations_coco.json"
EXPS = {n: load_exp_config(Path(REPO_DIR) / "configs" / f"{n}.yaml") for n in ['retinanet_baseline', 'retinanet_anchors', 'retinanet_assign', 'retinanet_tuned']}
RUN = list(EXPS)  # experiments this session may TRAIN; remove names to split the work over sessions
print("experiments:", RUN)

experiments: ['retinanet_baseline', 'retinanet_anchors', 'retinanet_assign', 'retinanet_tuned']


In [5]:
# Imports, GPU check and small helpers used below
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from src.utils import CLASS_NAMES, load_predictions
from src.eval import evaluate, per_class_table, evaluate_groups, clip_groups, density_groups
from src.viz import show_grid
from src import analysis as A
pd.set_option("display.width", 180)
RUN_GPU = torch.cuda.is_available()  # training / GPU-only cells are skipped on CPU
print("GPU:", torch.cuda.get_device_name(0) if RUN_GPU else "none -> training cells will be skipped")


def have(name, split="val"):
    return (WORK_DIR / "preds" / f"{name}_{split}.pkl").exists()


def compare(names, title=None):
    """COCO metrics of saved val predictions (same evaluator for every model)."""
    rows = {n: evaluate(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), GT_VAL, verbose=False)
            for n in names if have(n)}
    cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"] + [f"AP_{c}" for c in CLASS_NAMES]
    t = pd.DataFrame(rows, index=cols).T.round(4)
    if title and len(t):
        t.to_csv(RES_DIR / f"{title}.csv")
    return t


def speed_table(names):
    """Speed / training time from results/experiments.csv (last row per experiment)."""
    e = pd.read_csv(RES_DIR / "experiments.csv")
    e = e[e["experiment"].isin(names)].drop_duplicates("experiment", keep="last")
    return e[["experiment", "val_mAP", "AP50", "AP_small", "ms_per_img", "FPS", "train_time_min"]]

from src.dataset import TVDetectionDataset, load_rgb, read_yolo_labels
from src.models_tv import rpn_recall, anchor_assignment_stats
from src.train_tv import run_experiment, prepare_indices, load_trained, get_device
from src.anchors import gt_wh_at_input, anchor_wh, anchor_coverage
DEVICE = get_device()
RESULTS = {}


def run_tv(name, test=False, notes=""):
    """Train + evaluate one config (resumable; a finished run is only reloaded, never re-logged)."""
    done = (WORK_DIR / "tv" / name / "final.json").exists()
    if not done and not (RUN_GPU and name in RUN):
        print(f"{name}: skipped (no GPU or not in RUN)")
        return None
    r = run_experiment(EXPS[name], DATA_ROOT, WORK_DIR, test=test and RUN_GPU, notes=notes)
    r.pop("model", None)
    if RUN_GPU:
        torch.cuda.empty_cache()
    RESULTS[name] = r
    display(r["per_class"].set_index("class").T.round(3))
    display(r["per_clip"].round(3))
    return r


def smoke(name):
    """Whole pipeline on 2 frames per clip / 40 iterations: catches path, memory and API errors first."""
    if not RUN_GPU:
        print("smoke test skipped (no GPU)")
        return
    torch.cuda.reset_peak_memory_stats()
    run_experiment(EXPS[name], DATA_ROOT, WORK_DIR, smoke=True)
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    torch.cuda.empty_cache()

GPU: none -> training cells will be skipped


## 1. Anchors grounded in the object sizes (CPU)

In [6]:
# Frame indices (same as every model)
train_idx, val_epoch_idx, val_full_idx = prepare_indices(EXPS["retinanet_baseline"], DATA_ROOT)

[index] train: 56 clips, 4062 frames (step=6)
[index] val: 7 clips, 2846 frames (step=1)
[data] train 4062 | val per-epoch 714 | val final 2846


In [7]:
# Anchor coverage: 9 anchors / location (3 octave scales x 3 ratios) per level, GT rescaled to the network input
rows = []
for n in EXPS:
    m = EXPS[n]["model"]
    wh = gt_wh_at_input(train_idx, m["min_size_test"], m["max_size"])
    rows.append({"config": n, "input": f"{m['min_size_test']}/{m['max_size']}", "base_sizes": m["anchor_sizes"],
                 **anchor_coverage(wh, anchor_wh(m["anchor_sizes"], m["aspect_ratios"], m["anchor_scales"]))})
pd.DataFrame(rows).to_csv(RES_DIR / "retinanet_anchor_coverage.csv", index=False)
pd.DataFrame(rows).round(3)

,config,input,base_sizes,median_obj_px,cover_IoU>=0.5,cover_IoU>=0.7
0,retinanet_baseline,800/1333,"[32, 64, 128, 256, 512]",31.768,0.746,0.615
1,retinanet_anchors,800/1333,"[16, 32, 64, 128, 256]",31.768,0.982,0.937
2,retinanet_assign,800/1333,"[16, 32, 64, 128, 256]",31.768,0.982,0.937
3,retinanet_tuned,1024/1824,"[16, 32, 64, 128, 256]",43.385,0.994,0.963


In [8]:
# Anchor assignment on 30 val frames (no network): positives vs. negatives per image. Without a proposal stage
# every anchor is classified -> extreme foreground/background imbalance that focal loss has to handle.
stats = pd.DataFrame({n: anchor_assignment_stats(EXPS[n]["model"], val_full_idx, n_images=30) for n in EXPS}).T
stats.to_csv(RES_DIR / "retinanet_assignment_stats.csv")
stats

,anchors,gt,positives,negatives,between,neg_in_ignore,neg_per_pos,gt_with_1_anchor_%
retinanet_baseline,188748.0,37.3,704.9,186774.7,1268.4,1664.3,532.5,5.1
retinanet_anchors,188748.0,37.3,266.6,187980.2,501.1,1932.1,1400.3,11.4
retinanet_assign,188748.0,37.3,767.8,186743.7,1236.5,1923.2,493.3,5.5
retinanet_tuned,349992.0,37.3,738.7,348010.3,1243.0,3350.2,741.7,2.6


## 2. Smoke test (GPU, ~5 min)

In [9]:
# Smoke test of the whole pipeline with the heaviest config
smoke("retinanet_tuned")

smoke test skipped (no GPU)


## 3. Baseline and tuning (800/1333)

In [10]:
# Baseline: torchvision defaults (anchors 32-512 x 3 scales x 3 ratios, IoU 0.5 / 0.4, focal 0.25 / 2)
run_tv("retinanet_baseline", notes="torchvision defaults, ignore masked in focal loss");

retinanet_baseline: skipped (no GPU or not in RUN)


In [11]:
# Tuning 1: anchor base sizes 16-256 (EDA: median object 46 px, p5 20 px, shrunk x0.5-0.7 at 800/1333)
run_tv("retinanet_anchors", notes="anchor base sizes 16-256");

retinanet_anchors: skipped (no GPU or not in RUN)


In [12]:
# Tuning 2: positive / negative IoU thresholds 0.4 / 0.3 (more positive anchors for small boxes)
run_tv("retinanet_assign", notes="anchors 16-256 + IoU fg/bg 0.4/0.3");

retinanet_assign: skipped (no GPU or not in RUN)


## 4. Final one-stage anchor-based model (1024/1824) + test submission

In [13]:
# Tuned RetinaNet; also predicts the test set -> WORK_DIR/submissions/retinanet_tuned.csv (validated)
run_tv("retinanet_tuned", test=True, notes="anchors 16-256, IoU 0.4/0.3, 1024/1824 + jitter");

retinanet_tuned: skipped (no GPU or not in RUN)


## 5. Comparison with the two-stage detector

In [14]:
# Initial and tuned RetinaNet vs. the tuned Faster R-CNN (same protocol) + speed
names = list(EXPS) + ["frcnn_baseline", "frcnn_tuned"]
display(compare(names, "retinanet_comparison"))
speed_table(names)

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR,AP_person,AP_bicycle,AP_car,AP_van,AP_truck,AP_tricycle,AP_bus,AP_motor


,experiment,val_mAP,AP50,AP_small,ms_per_img,FPS,train_time_min


In [15]:
# AP by object size (COCO buckets) and by scene density (number of GT objects per frame)
names = [n for n in ["retinanet_baseline", "retinanet_tuned", "frcnn_tuned"] if have(n)]
if names:
    display(compare(names, "anchor_based_by_size")[["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]])
    dens = density_groups(GT_VAL)
    t = pd.concat([evaluate_groups(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), GT_VAL, dens)
                   .assign(model=n) for n in names])
    t.to_csv(RES_DIR / "anchor_based_by_size_by_density.csv", index=False)
    display(t.pivot_table(index="model", columns="group", values="mAP", sort=False).round(4))
else:
    print("no saved predictions yet")

no saved predictions yet


In [16]:
# Without proposals: false positives per image and recall per size at score >= 0.3 (one-stage vs. two-stage)
gt_val = A.gt_from_coco(GT_VAL)
rows = []
for n in ["retinanet_tuned", "frcnn_tuned"]:
    if have(n):
        fr, g, fp = A.analyse_predictions(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), gt_val, conf=0.3)
        rows.append({"model": n, "FP_per_img": len(fp) / len(fr), "wrong_cls_per_img": fr["wrong_cls"].mean(),
                     **{f"recall_{b}": g.loc[g["bucket"] == b, "matched"].mean() for b in ["small", "medium", "large"]}})
pd.DataFrame(rows).round(3) if rows else print("no saved predictions yet")

no saved predictions yet


In [17]:
# Recall (score >= 0.3, IoU >= 0.5) vs. object size for each model: where are small objects lost?
names = [n for n in ["retinanet_baseline", "retinanet_tuned", "frcnn_tuned"] if have(n)]
if names:
    gt_val = A.gt_from_coco(GT_VAL)
    bins = [0, 8, 12, 16, 24, 32, 48, 64, 96, 1e9]
    fig, ax = plt.subplots(figsize=(7, 4))
    for n in names:
        _, g, _ = A.analyse_predictions(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), gt_val, conf=0.3)
        r = g.groupby(pd.cut(g["size"], bins), observed=False)["matched"].mean()
        ax.plot(range(len(r)), r.values, marker="o", label=n)
    ax.set_xticks(range(len(bins) - 1), [f"{int(a)}-{int(b) if b < 1e8 else '+'}" for a, b in zip(bins[:-1], bins[1:])], rotation=30)
    ax.set(xlabel="object size sqrt(w*h) [px]", ylabel="recall", ylim=(0, 1)); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.savefig(FIG_DIR / "retinanet_recall_vs_size.png", dpi=120, bbox_inches="tight"); plt.show()